# 03 – EfficientNet-B0 Transfer Learning

**Stage 1:** freeze backbone, train head (lr 1e-3, 3 epochs)  
**Stage 2:** unfreeze last 2–3 blocks, lr 1e-4, 5 epochs  

Augmentation: random horizontal flip only. Early stopping on val loss.

In [ ]:
# --- Colab setup (run first) ---
# Runtime → Change runtime type → T4 GPU
!pip -q install kagglehub grad-cam

from pathlib import Path
from google.colab import drive
import kagglehub

# Everything we produce goes to Drive so it survives Colab restarts
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/ai-image-detector")
MODELS = OUT / "models"; REPORT = OUT / "report"
MODELS.mkdir(parents=True, exist_ok=True); REPORT.mkdir(parents=True, exist_ok=True)

# Downloads CIFAKE once (~100 MB); contains train/{REAL,FAKE} and test/{REAL,FAKE}
DATA_ROOT = Path(kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images"))
print("Data:", DATA_ROOT, "| Outputs:", OUT)

In [ ]:
import os, random, time, copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

BATCH = 64
IMG_SIZE = 224

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = datasets.ImageFolder(DATA_ROOT / "train", transform=train_tf)
n_val = int(0.1 * len(full_train))
n_train = len(full_train) - n_val
train_ds, val_ds = random_split(full_train, [n_train, n_val],
                                generator=torch.Generator().manual_seed(SEED))
# val must use eval transform – re-wrap
val_ds.dataset = copy.copy(full_train)
val_ds.dataset.transform = eval_tf

test_ds = datasets.ImageFolder(DATA_ROOT / "test", transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)
print(f"Train {n_train:,} | Val {n_val:,} | Test {len(test_ds):,}")

In [ ]:
def build_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features, 1),
    )
    return model

model = build_model().to(DEVICE)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
criterion = nn.BCEWithLogitsLoss()

def run_epoch(loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, preds, targets = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.float().to(DEVICE)
        y = 1.0 - y  # FAKE=0 → AI=1
        with torch.set_grad_enabled(train):
            logits = model(x).squeeze(1)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * x.size(0)
        preds.append(torch.sigmoid(logits).detach().cpu())
        targets.append(y.cpu())
    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    acc = accuracy_score(targets, preds >= 0.5)
    return total_loss / len(loader.dataset), acc, preds, targets

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val = float("inf")
patience, wait = 3, 0

In [ ]:
# ---- Stage 1: freeze backbone, train head ----
for p in model.features.parameters():
    p.requires_grad = False
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

print("=== Stage 1: head only ===")
for epoch in range(1, 4):
    t0 = time.time()
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, optimizer)
    va_loss, va_acc, _, _ = run_epoch(val_loader)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(va_acc)
    print(f"S1 E{epoch} | train {tr_loss:.4f}/{tr_acc:.3f} | val {va_loss:.4f}/{va_acc:.3f} | {time.time()-t0:.0f}s")
    if va_loss < best_val:
        best_val = va_loss; wait = 0
        torch.save({"model_state_dict": model.state_dict()}, MODELS / "effnet_b0_best.pt")
        print("  → saved")
    else:
        wait += 1

In [ ]:
# ---- Stage 2: unfreeze last 3 blocks ----
# EfficientNet-B0 features is a Sequential of 8 stages (0..7)
for name, child in model.features.named_children():
    idx = int(name)
    for p in child.parameters():
        p.requires_grad = idx >= 5  # last 3 blocks (5,6,7)

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

print("=== Stage 2: fine-tune last blocks ===")
for epoch in range(1, 6):
    t0 = time.time()
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, optimizer)
    va_loss, va_acc, _, _ = run_epoch(val_loader)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(va_acc)
    print(f"S2 E{epoch} | train {tr_loss:.4f}/{tr_acc:.3f} | val {va_loss:.4f}/{va_acc:.3f} | {time.time()-t0:.0f}s")
    if va_loss < best_val:
        best_val = va_loss; wait = 0
        torch.save({"model_state_dict": model.state_dict()}, MODELS / "effnet_b0_best.pt")
        print("  → saved")
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping"); break

In [ ]:
# Test evaluation
ckpt = torch.load(MODELS / "effnet_b0_best.pt", map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
_, test_acc, preds, targets = run_epoch(test_loader)
prec, rec, f1, _ = precision_recall_fscore_support(targets, preds >= 0.5, average="binary")
auc = roc_auc_score(targets, preds)
cm = confusion_matrix(targets, preds >= 0.5)

print(f"Test Acc  {test_acc:.4f}")
print(f"Precision {prec:.4f}  Recall {rec:.4f}  F1 {f1:.4f}  AUC {auc:.4f}")
print("CM:\n", cm)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].axvline(2.5, color="gray", ls="--", alpha=0.5)
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout()
plt.savefig(REPORT / "effnet_curves.png", dpi=150)
plt.show()